# Basline Impelementations

In [4]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt  
import matplotlib.ticker as mticker     
import seaborn as sns 
import math 
import itertools 
from evaluation_code import rmse

In [5]:
# first lets load in our datasets 
train= pd.read_csv('data/train.csv') # user-movie interaction file with ratings we can train on
test_pairs= pd.read_csv('data/test_pairs.csv') # user-movie pairs we need to get ratings for, can also thing of predicting rank
movies = pd.read_csv('data/movies.csv')  # movie metadata
users = pd.read_csv('data/users.csv')  # user metadata


# Create a Validation Split
Before actually starting with model building lets build a validation split so we can test the validity of the our model similar to how the the assignment would be graded on. For each user we will sort ratings by timestamp, we will take the oldest 80% as the training data and then we will be using the newest 20% as the validation. 

In [6]:
def make_val_split(df, frac=0.20):
    """
    Temporal validation 20% split, can adjust based on the frac parameter 
    Sort ratings from old to new and hold out tail frac percent 
    """
    train_rows = []  # collect individual user training part 
    val_rows   = [] # collect individual validation part

    for user_id, group in df.groupby("user_id"):  # one group for each user_id, dont use actual user_id just want observations with that user_id       
        group = group.sort_values("timestamp") # sort it
        n = len(group) # how many ratings total user has given 
        n_val = math.ceil(frac * n) # how many validation observations will be used, use ciel to round up, defensive programming
        train_rows.append(group.iloc[:n - n_val]) # older 80% used for training using slicing
        val_rows.append(group.iloc[n - n_val:])  # newer 20% used for training using slicing

    my_train = pd.concat(train_rows).reset_index(drop=True)  # add to one dataframe for train
    my_val   = pd.concat(val_rows).reset_index(drop=True)   # one dataframe for val
    return my_train, my_val

In [7]:
training_set, validation_set = make_val_split(train) 

In [8]:
training_set

,user_id,item_id,rating,timestamp
0,1,3186,4,978300019
1,1,1721,4,978300055
2,1,1270,5,978300055
3,1,1022,5,978300055
4,1,2340,3,978300103
...,...,...,...,...
635718,6040,1213,4,957716861
635719,6040,1219,5,957716861
635720,6040,1929,5,957716861
635721,6040,1104,5,957716904


In [9]:
validation_set

,user_id,item_id,rating,timestamp
0,1,3114,4,978302174
1,1,2791,4,978302188
2,1,2321,3,978302205
3,1,1029,5,978302205
4,1,594,4,978302268
...,...,...,...,...
162030,6040,2612,5,960971797
162031,6040,1449,3,960971857
162032,6040,2303,5,960971857
162033,6040,3504,4,960971857


In [10]:
# lets also extract the columns that the RMSE will need for evaluation, we will compare  val_truth to predicted ratings
val_truth = validation_set[["user_id", "item_id", "rating"]]


### Baseline 1: Global Mean Baseline
The simplest baseline is predictig the same number for each user and movie pair. This number is just the average of all the training ratings

In [11]:
# set up the simple validation global predictions df to input into evalusation code rmse
val_predictions_mean = validation_set[["user_id", "item_id"]].copy() 
val_predictions_mean["predicted_rating"] = float(training_set["rating"].mean())

rmse_global_mean = rmse(val_truth, val_predictions_mean)
print(f" Global Mean Baseline RMSE: {rmse_global_mean}")

 Global Mean Baseline RMSE: 1.1200808008497214


### Baseline 2: User Mean Baseline
For each user we just predict their rating to be the average of the ratings that they have given. This is a slightly better improvement than the global mean as it does consider user by user signal. 

In [12]:
user_mean = training_set.groupby('user_id')['rating'].mean() # compute each users average rating

# Now set up these predicted ratings with validation set 
val_predictions_user = validation_set[['user_id', 'item_id']].copy()
val_predictions_user["predicted_rating"] = val_predictions_user["user_id"].map(user_mean) # look up each user's mean rating

rmse_user = rmse(val_truth, val_predictions_user)
print(f" User Mean Baseline RMSE: {rmse_user}")

 User Mean Baseline RMSE: 1.0466261860648265


### Baseline 3: Item Mean Baseline
For each  user we just predict their rating to be the average of the rating of the item that they are paired with. This is a slightly better improvement than the user mean as it does considers the item rating which tends to be a universal view on the movie than user's tastes.

In [13]:
item_mean = training_set.groupby('item_id')['rating'].mean() # compute each users average rating

# Now set up these predicted ratings with validation set 
val_predictions_item = validation_set[['user_id', 'item_id']].copy()
val_predictions_item["predicted_rating"] = val_predictions_item["item_id"].map(item_mean) # look up each item's mean rating
val_predictions_item["predicted_rating"] = val_predictions_item["predicted_rating"].fillna(training_set["rating"].mean()) # for items with not reviews just

rmse_item = rmse(val_truth, val_predictions_item)
print(f" Item Mean Baseline RMSE: {rmse_item}")

 Item Mean Baseline RMSE: 0.985298687310287


### Baseline 4: Bias Model
We will now be using a combined bias model that combines both aspects of the users signal and the items quality. We can also use reguralization do deal with sparse data or cold start items.
The general formula is mu + user bias + item bias, where user bias tells us how much the user tends to rate above or below mean, item ias is how much some movie is rated above or below mean. We will also.use reguralization to account for the case where a movie only has 1 rating of 5 stars this gives a high item bias that dominates. To account for this when calculating the item bias we will add  some lambda in the denominator b_i = sum(residuals) / count --> b_i = sum(residuals) / (count + λ_i) so in cases where rating count for item is large the lambda is negligble and is review count is low the lambda shrinks the bias and the predicted. rating falls around the global mean. \
\
To calculate user bias and item bias you need either or so to fix this we will start with initializing both biases at 0. Then iterate through each bias 20 times similar to weight update logic in SGD or BGD.

Article that describes this method:
https://blog.csdn.net/weixin_47966242/article/details/125859085

In [14]:
def fit_bias_model(train_df, mu, lambda_u=10.0, lambda_i=25.0, n_iter=20): # we have set lambdas for reguralization strength for each respective bias 
    """
    fit reguralized user and item bias using alternate updates over n_iter iterations
    Formula used: 
     b_i = sum_u(r_ui - mu - b_u) / (|count of ratings item i has recieved| + lambda_i)
     b_u = sum_i(r_ui - mu - b_i) / (|count of ratings user u has given| + lambda_u)
    """
    b_u = pd.Series(0.0, index=train_df["user_id"].unique()) # initialize at 0.
    b_i = pd.Series(0.0, index=train_df["item_id"].unique())

    df = train_df.copy()

    for i in range(n_iter): # n_iter is set at 20 so we run through 20 iterations 
        df["b_u"] = df["user_id"].map(b_u).fillna(0.0)
        df["residual_for_item"] = df["rating"] - mu - df["b_u"] # we get residual for each rating and  global mean subtracted by current user bias guess (start at 0)
        item_stats = df.groupby("item_id")["residual_for_item"].agg(["sum", "count"])  #aggregate sum of residuals and count of ratings per item for reguralized bias formula 
        b_i = item_stats["sum"] / (item_stats["count"] + lambda_i) # the reguralization formula

        # we are doing the same thing but for the item bias
        df["b_i"] = df["item_id"].map(b_i).fillna(0.0) 
        df["residual_for_user"] = df["rating"] - mu - df["b_i"]
        user_stats = df.groupby("user_id")["residual_for_user"].agg(["sum", "count"])
        b_u = user_stats["sum"] / (user_stats["count"] + lambda_u)

    return b_u, b_i


def predict_bias(pairs, mu, b_u, b_i):
    """
    predict ratings using actual bias formula μ + b_u + b_i

    """
    out = pairs[["user_id", "item_id"]].copy() # target pairs
    out["bu"] = out["user_id"].map(b_u).fillna(0.0) # look up the user bias and if not there fill with na 
    out["bi"] = out["item_id"].map(b_i).fillna(0.0) # look up item bias fill with 0 
    out["predicted_rating"] = (mu + out["bu"] + out["bi"]).clip(1.0, 5.0) # calculate predicted rating, clip from 1 to 5 to make sure doesnt exceed
    return out[["user_id", "item_id", "predicted_rating"]] # return what we need

In [15]:
global_mean_full = float(training_set["rating"].mean())
                    
b_u, b_i = fit_bias_model(training_set, global_mean_full)   # fit with default lambda values

In [16]:
b_i

item_id
1       0.624467
2      -0.313340
3      -0.398083
4      -0.428661
5      -0.415941
          ...   
3948    0.187143
3949    0.630184
3950   -0.020211
3951    0.243473
3952    0.247001
Length: 3635, dtype: float64

In [17]:

prediction_bias = predict_bias(validation_set, global_mean_full, b_u, b_i)  # predict on validation pairs



In [18]:
prediction_bias

,user_id,item_id,predicted_rating
0,1,3114,4.456994
1,1,2791,4.231945
2,1,2321,3.971630
3,1,1029,3.930674
4,1,594,4.057790
...,...,...,...
162030,6040,2612,3.422909
162031,6040,1449,3.678332
162032,6040,2303,3.465147
162033,6040,3504,3.608306


In [19]:
rmse_bias_default = rmse(val_truth, prediction_bias)   # score against ground truth


In [20]:
print(f" bias model val RMSE : {rmse_bias_default:.4f}")

 bias model val RMSE : 0.9181


In [21]:
### we can do some grid search on the lambda to get a better rmse

lambda_u_grid = [1, 5, 10, 15, 25] # our values of u lambda to try
lambda_i_grid = [1, 5, 10, 25, 50] # our values of i lamba to try 

tuning_results = [] # to collect different combos lambdas 

for lu, li in itertools.product(lambda_u_grid, lambda_i_grid): # trying every combination 
    b_u, b_i = fit_bias_model(training_set, global_mean_full, lambda_u=lu, lambda_i=li)
    predictions = predict_bias(validation_set, global_mean_full, b_u, b_i)
    scores = rmse(val_truth, predictions)
    tuning_results.append({"lambda_u": lu, "lambda_i": li, "val_rmse": scores})

results_df = pd.DataFrame(tuning_results)
results_df = results_df.sort_values("val_rmse")

In [22]:
results_df.head(10)


,lambda_u,lambda_i,val_rmse
5,5,1,0.912953
6,5,5,0.912979
10,10,1,0.913139
11,10,5,0.913182
0,1,1,0.913198
1,1,5,0.913225
15,15,1,0.913579
16,15,5,0.913637
7,5,10,0.913861
2,1,10,0.914047


In [23]:
best_lu = results_df.iloc[0]["lambda_u"]
best_li = results_df.iloc[0]["lambda_i"]
best_bias_rmse = float(results_df.iloc[0]["val_rmse"])

print(f"best lambda_u: {best_lu}, best lambda_i: {best_li}, val RMSE: {best_bias_rmse:.4f}")

best lambda_u: 5.0, best lambda_i: 1.0, val RMSE: 0.9130


In [24]:

comparison = pd.DataFrame({
    "Model":    ["Global Mean", "User Mean", "Item Mean", f"Bias Model (lu={best_lu}, li={best_li})"],
    "Val RMSE": [rmse_global_mean, rmse_user, rmse_item, best_bias_rmse]
})
print(comparison.to_string(index=False))


                      Model  Val RMSE
                Global Mean  1.120081
                  User Mean  1.046626
                  Item Mean  0.985299
Bias Model (lu=5.0, li=1.0)  0.912953


In [25]:
global_mean_full = float(train["rating"].mean())

b_u_full, b_i_full = fit_bias_model(train, global_mean_full, lambda_u=best_lu, lambda_i=best_li)
preds_bias_test = predict_bias(test_pairs, global_mean_full, b_u_full, b_i_full)

preds_bias_test.to_csv("predictions_bias.csv", index=False)